# Baseline, scrubbed SFT, DPO, and GRPO — run one method at a time

**For now, run setup and the Baseline cell.** `RUN_CONDITION = "baseline"` also makes Run All skip SFT, DPO, and GRPO. Later, change that setting and run the corresponding method cell. Nothing has been trained as part of these code edits.

All methods append epoch diagnostics to **`results/validation_loss_graph.csv`**. Keep the same `EXPERIMENT_ID` when returning to run another method. A new comparison or rerun needs a new experiment ID; prior checkpoints and CSV rows are preserved.

Baseline and scrubbed SFT start independently from GPT-2. DPO and GRPO each start from this experiment's **best-validation baseline checkpoint**, with an independent frozen copy as their reference. DPO does not start from GRPO, and GRPO does not start from DPO. Baseline must finish before either alignment method.

Run notebook 01 first if the revised data has not been generated. Defaults are 1,000 report tasks plus 30 training utility questions per method; validation is the same 40 clean reports plus 30 validation utility questions. Test data is never used for training, rewards, or checkpoint selection.


In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src" / "train_study.py").is_file(), "Open this notebook from the project root or notebooks folder."
assert (PROJECT_ROOT / "data" / "synthetic" / "training_conversations.jsonl").is_file(), "Run notebook 01 first."
print(f"Project root: {PROJECT_ROOT}")
# Uncomment if dependencies are missing:
# subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)


## Shared settings

Change `RUN_CONDITION` to `"sft_filtered"`, `"dpo"`, or `"grpo"` when you are ready for that method. Keep `EXPERIMENT_ID`, dataset, seed, report limit, and maximum length unchanged across this comparison. The trainer records and checks these settings.

Five epochs are a starting point. Epoch 0 records initialization; epochs 1–5 record results after training and are eligible for checkpoint selection. For DPO/GRPO, an epoch counts a pass over training tasks **after baseline training**, so the x-axis represents the current stage, not equal total compute.

GRPO samples four answers per prompt and makes two updates per rollout group. It will take substantially longer than baseline/SFT. Its training-only reward combines reference-answer word overlap with penalties for exact private-value disclosure. Inspect the reward and zero-variance-group columns; this heuristic does not replace held-out privacy testing or human utility scoring.


In [ ]:
RUN_CONDITION = "baseline"  # Run only this method; other method cells skip.
EXPERIMENT_ID = "four-method-v1"  # Keep this exact ID for all four methods.
MODEL_NAME = "gpt2"
SEED = 42
TRAIN_EXAMPLES = None  # All 1,000 reports + 30 utility examples.
MAX_LENGTH = 384
BATCH_SIZE = 1
ACCUMULATION_STEPS = 8
EPOCHS = {"baseline": 5, "sft_filtered": 5, "dpo": 5, "grpo": 5}
LEARNING_RATES = {"baseline": 5e-5, "sft_filtered": 5e-5, "dpo": 5e-6, "grpo": 1e-6}
DPO_BETA = 0.1
GRPO_BETA = 0.04
GRPO_GROUP_SIZE = 4
GRPO_UPDATES = 2
GRPO_CLIP = 0.2
GRPO_MAX_NEW_TOKENS = 96

CONDITIONS = ("baseline", "sft_filtered", "dpo", "grpo")
assert RUN_CONDITION in CONDITIONS
assert Path(EXPERIMENT_ID).name == EXPERIMENT_ID and EXPERIMENT_ID not in {".", ".."}
METRICS_PATH = PROJECT_ROOT / "results" / "validation_loss_graph.csv"
EXPERIMENT_DIR = PROJECT_ROOT / "models" / "four-method-study" / EXPERIMENT_ID
print(f"Selected method: {RUN_CONDITION}")
print(f"Shared CSV: {METRICS_PATH}")
print(f"Experiment checkpoints: {EXPERIMENT_DIR}")


## Shared runner

The four cells below invoke the implementation in `src/train_study.py` in separate Python processes so model memory is released when each run ends. Each invocation trains **one** condition and appends to the shared CSV; no earlier conditions are retrained.

The CSV's `train_loss_assistant_tokens` always scores the same clean training set, and `validation_loss_assistant_tokens` scores the same clean validation set. Those are comparable cross-entropy diagnostics for every method. `objective_loss` is the method's actual optimization loss (baseline CE, SFT CE, DPO preference loss, or GRPO objective); do not compare that column across different methods.


In [ ]:
def run_method(condition):
    if RUN_CONDITION != condition:
        print(f"Skipped {condition}; RUN_CONDITION is {RUN_CONDITION!r}.")
        return
    command = [
        sys.executable, "-u", str(PROJECT_ROOT / "src" / "train_study.py"),
        "--condition", condition,
        "--experiment-dir", str(EXPERIMENT_DIR),
        "--data-dir", str(PROJECT_ROOT / "data" / "synthetic"),
        "--metrics-csv", str(METRICS_PATH),
        "--model-name", MODEL_NAME,
        "--seed", str(SEED), "--epochs", str(EPOCHS[condition]),
        "--batch-size", str(BATCH_SIZE),
        "--accumulation-steps", str(ACCUMULATION_STEPS),
        "--max-length", str(MAX_LENGTH),
        "--learning-rate", str(LEARNING_RATES[condition]),
    ]
    if TRAIN_EXAMPLES is not None:
        command.extend(["--max-train-examples", str(TRAIN_EXAMPLES)])
    if condition == "dpo":
        command.extend(["--beta", str(DPO_BETA)])
    if condition == "grpo":
        command.extend([
            "--beta", str(GRPO_BETA), "--group-size", str(GRPO_GROUP_SIZE),
            "--grpo-updates", str(GRPO_UPDATES), "--clip", str(GRPO_CLIP),
            "--max-new-tokens", str(GRPO_MAX_NEW_TOKENS),
        ])
    print(f"Starting only {condition}; appending to {METRICS_PATH}", flush=True)
    process = subprocess.Popen(
        command, cwd=PROJECT_ROOT, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1,
    )
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
        return_code = process.wait()
    except BaseException:
        process.terminate()
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        raise
    finally:
        process.stdout.close()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)


## 1. Baseline — run this now

Supervised fine-tuning on the 1,030 exposed training conversations. Personal details are deliberately present in 800 reports. The best checkpoint is saved in this experiment's `baseline/` folder.


In [ ]:
run_method("baseline")


## 2. Scrubbed SFT — run later

Set `RUN_CONDITION = "sft_filtered"` in the settings cell, rerun settings, and run this cell. This starts from GPT-2 and trains on all 1,030 paired clean conversations. It does not continue training the baseline.


In [ ]:
run_method("sft_filtered")


## 3. DPO — run later, after baseline completes

Set `RUN_CONDITION = "dpo"`, rerun settings, and run this cell. It loads the saved baseline and optimizes the [DPO preference objective](https://arxiv.org/abs/2305.18290) against a frozen baseline reference.

The runner builds 1,030 preference pairs exclusively from the paired training data. The prompt is the exposed user question; the chosen answer retains the useful clean service response. For PII reports, the rejected answer reveals the private fields. For non-PII reports and utility questions, the rejected answer is a blanket refusal, encouraging useful answers rather than refusal-only behavior. These pairs are built in memory; the older `dpo_training_pairs.jsonl` is not used by this runner.


In [ ]:
run_method("dpo")


## 4. GRPO — run later, after baseline completes

Set `RUN_CONDITION = "grpo"`, rerun settings, and run this cell. It independently loads the saved baseline and applies [outcome-supervised GRPO](https://arxiv.org/abs/2402.03300): grouped sampled answers, normalized within-group rewards, clipped policy ratios, and a KL penalty against a frozen baseline reference.

The training reward is reference-answer content-word F1 minus 2 per exact private-field disclosure from the selected training records. Empty responses receive −1; explicit blanket refusals receive no utility credit. This reward is an experimental proxy with limited semantic understanding. It never reads validation/test probes or held-out utility answers. Zero reward variance gives zero relative advantage; that fraction is logged for diagnosis.


In [ ]:
run_method("grpo")


## Visualize whichever methods have finished

Run this after baseline, then rerun it after each additional method. It filters the shared CSV to the current experiment and leaves unfinished panels empty. Stars mark the lowest validation loss among trained epochs. Epoch-zero points show initialization.

A decreasing validation loss indicates better prediction of clean reference answers; it does not establish privacy protection or useful generated responses. DPO and GRPO curves start after the baseline stage. Compare held-out privacy and utility separately once training settings are fixed.


In [ ]:
import csv
import matplotlib.pyplot as plt

with METRICS_PATH.open(newline="", encoding="utf-8") as metrics_file:
    metric_rows = [row for row in csv.DictReader(metrics_file) if row["experiment_id"] == EXPERIMENT_ID]
if not metric_rows:
    print("No metrics for this experiment yet. Run the baseline cell first.")
else:
    if len({row["dataset_fingerprint"] for row in metric_rows}) != 1:
        raise ValueError("This experiment's CSV rows refer to different datasets.")
    labels = {"baseline": "Baseline", "sft_filtered": "Scrubbed SFT", "dpo": "DPO", "grpo": "GRPO"}
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True)
    comparison_fig, comparison_axis = plt.subplots(figsize=(8, 5))
    best_epochs = {}
    for axis, condition in zip(axes.flat, CONDITIONS):
        rows = sorted((row for row in metric_rows if row["condition"] == condition), key=lambda row: int(row["epoch"]))
        axis.set_title(labels[condition])
        if not rows:
            axis.text(0.5, 0.5, "Not run yet", ha="center", va="center", transform=axis.transAxes)
            continue
        epochs = [int(row["epoch"]) for row in rows]
        if len(epochs) != len(set(epochs)):
            raise ValueError(f"Duplicate epoch rows for {condition}; select a distinct experiment ID.")
        train_loss = [float(row["train_loss_assistant_tokens"]) for row in rows]
        validation_loss = [float(row["validation_loss_assistant_tokens"]) for row in rows]
        axis.plot(epochs, train_loss, marker="o", label="Common clean training CE")
        axis.plot(epochs, validation_loss, marker="o", label="Validation CE")
        comparison_axis.plot(epochs, validation_loss, marker="o", label=labels[condition])
        trained = [i for i, epoch in enumerate(epochs) if epoch > 0]
        if trained:
            best = min(trained, key=lambda i: validation_loss[i])
            best_epochs[condition] = {"epoch": epochs[best], "validation_loss": validation_loss[best]}
            axis.scatter([epochs[best]], [validation_loss[best]], marker="*", s=150, color="tab:red", zorder=3, label="Best trained epoch")
        axis.set_xlabel("Epoch within this training stage (0 = initialization)")
        axis.set_ylabel("Assistant-token cross entropy")
        axis.set_xticks(epochs)
        axis.grid(True, alpha=0.3)
        axis.legend(fontsize=8)
    fig.suptitle(f"Training and validation loss — {EXPERIMENT_ID}")
    fig.tight_layout()
    plot_path = METRICS_PATH.with_suffix(".png")
    fig.savefig(plot_path, dpi=160, bbox_inches="tight")
    comparison_axis.set(title=f"Shared validation set — {EXPERIMENT_ID}", xlabel="Epoch within this training stage", ylabel="Assistant-token cross entropy")
    comparison_axis.grid(True, alpha=0.3)
    comparison_axis.legend()
    comparison_fig.tight_layout()
    comparison_path = METRICS_PATH.with_name("validation_loss_graph_comparison.png")
    comparison_fig.savefig(comparison_path, dpi=160, bbox_inches="tight")
    plt.show()
    print("Lowest validation-loss trained epochs:", best_epochs)
    print("Saved plots:", plot_path, comparison_path)


## Later: evaluate generated responses

After choosing settings on validation, use `src/evaluate_models.py --run-dir <experiment folder> --conditions baseline sft_filtered dpo grpo --split validation --output-dir <new results folder>`. Select only completed methods. Omitting `--conditions` evaluates whichever condition folders are present, so baseline can also be evaluated alone. Utility still requires manual rubric scoring. Reserve the test split for final reporting.
